# 🏥 PatientDoctorBridge — Kaggle Gemma 4 Impact Challenge

> **A privacy-first, local-only multilingual medical interpreter powered exclusively by Gemma 4.**

---

## What this notebook demonstrates

PatientDoctorBridge breaks the language barrier between non-English-speaking patients and doctors in Indian clinics. It runs **100% locally** — no cloud calls, no data leaves the device.

| Scenario | Description | Gemma 4 model |
|---|---|---|
| **Bridge Translation** | Patient speech (Hindi/Telugu/Kannada/Tamil) ↔ Doctor English | `gemma4:e2b` |
| **Emergency Triage** | Structured JSON extraction from patient speech | `gemma4:e4b` |
| **Prescription OCR** | Handwritten prescription → structured medicine list | `gemma4:e4b` (vision) |
| **Emergency Reassurance** | Fast English phrase → patient language translation | `gemma4:e2b` |

### Architecture
```
Patient speaks → [Whisper ASR] → transcript → [Gemma 4] → Doctor English
Doctor speaks  → [Whisper ASR] → transcript → [Gemma 4] → Patient language
Prescription image            →             → [Gemma 4 vision] → JSON
```

All inference runs via **Ollama** using:
- `gemma4:e2b` — FAST_TRANSLATION mode (Patient↔Doctor, Reassurance)
- `gemma4:e4b` — REASONING_EXTRACTION mode (Triage, OCR)

> 📌 **Judge instructions**: Run all cells top-to-bottom. The final cell opens the full interactive UI in your browser via a public URL. Text-based demos run inline without audio.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 1 — Hardware Detection
# Auto-selects GPU (float16) or CPU (int8) for all inference.
# ═══════════════════════════════════════════════════════════════
import os
import platform
import subprocess
import sys

print("=" * 60)
print("PatientDoctorBridge — Hardware Detection")
print("=" * 60)
print(f"Python : {sys.version.split()[0]}")
print(f"OS     : {platform.system()} {platform.release()}")
print(f"CPUs   : {os.cpu_count()}")
print()

# --- CUDA check ---------------------------------------------------------
has_gpu = False
gpu_name = "None"
gpu_memory_gb = 0.0

try:
    import torch
    if torch.cuda.is_available():
        has_gpu = True
        gpu_name = torch.cuda.get_device_name(0)
        gpu_memory_gb = round(
            torch.cuda.get_device_properties(0).total_memory / (1024**3), 1
        )
        print(f"✅ GPU detected : {gpu_name} ({gpu_memory_gb} GB VRAM)")
        print(f"   CUDA version : {torch.version.cuda}")
        print("   Whisper will run on GPU with float16 — fast inference ⚡")
    else:
        print("⚠️  No CUDA GPU detected — running on CPU (int8). Slower but functional.")
except ImportError:
    print("⚠️  torch not installed — CPU mode.")

# --- TPU check (informational only) ------------------------------------
try:
    result = subprocess.run(["ls", "/dev/accel0"], capture_output=True)
    if result.returncode == 0:
        print("ℹ️  TPU device found but NOT used for Whisper/Ollama (JAX only).")
except Exception:
    pass

print()
print(f"Device    : {'CUDA (GPU)' if has_gpu else 'CPU'}")
print(f"GPU Name  : {gpu_name}")
print(f"VRAM      : {gpu_memory_gb} GB")
print("=" * 60)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 2 — System Setup (Ollama + audio libraries)
# ═══════════════════════════════════════════════════════════════
import os
import subprocess


def run(cmd, **kw):
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if result.stdout.strip():
        print(result.stdout.strip()[-500:])
    if result.returncode != 0 and result.stderr.strip():
        print("STDERR:", result.stderr.strip()[-300:])
    return result.returncode

def ollama_works():
    try:
        r = subprocess.run(["ollama", "--version"],
                           capture_output=True, text=True, timeout=10)
        return r.returncode == 0
    except Exception:
        return False

if not ollama_works():
    print("Installing Ollama prerequisites…")
    # MUST refresh package cache first — stale cache causes 404 on version URLs.
    # curl is already present in Kaggle; only zstd is missing (required by the
    # Ollama install script for tarball extraction since v0.4+).
    run("apt-get update -qq")
    run("apt-get install -y -q --fix-missing zstd")

    print("\nInstalling Ollama…")
    run("rm -f /usr/local/bin/ollama")
    run("curl -fsSL https://ollama.com/install.sh | sh")

    if ollama_works():
        r = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
        print("✅ Ollama installed:", r.stdout.strip())
    else:
        print("❌ Ollama still not working — see output above")
        raise RuntimeError("Ollama installation failed. Cannot continue.")
else:
    r = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
    print("✅ Ollama already installed:", r.stdout.strip())

# System audio/video libraries needed by faster-whisper and soundfile
run("apt-get install -y -q ffmpeg libsndfile1 2>/dev/null || true")

print("\n✅ System dependencies ready.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 3 — Python Package Installation
# ═══════════════════════════════════════════════════════════════
import subprocess
import sys


def pip(*pkgs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + list(pkgs)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print("pip error:", r.stderr[-300:])
    else:
        print(f"✅ Installed: {', '.join(pkgs)}")

pip("faster-whisper==1.1.1")
pip("Flask==3.1.1", "Werkzeug==3.1.3")
pip("ollama==0.4.7")
pip("rich==14.0.0", "typer==0.15.1")
pip("soundfile==0.13.1")
pip("pyngrok==7.2.2")
pip("numpy>=1.24")

print("\n✅ All Python packages ready.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 4 — Project Path Setup
# ═══════════════════════════════════════════════════════════════
import os
import pathlib
import sys

# ── Locate the project source ────────────────────────────────
# Priority 1: dataset mounted at /kaggle/input/patient-doctor-bridge/
# Priority 2: current working directory (if running locally)
CANDIDATE_PATHS = [
    "/kaggle/input/patient-doctor-bridge",
    "/kaggle/input/patientdoctorbridge",
    str(pathlib.Path.cwd()),
    str(pathlib.Path.cwd().parent),
]

PROJECT_ROOT = None
for p in CANDIDATE_PATHS:
    if os.path.isfile(os.path.join(p, "core", "engine.py")):
        PROJECT_ROOT = p
        break

if PROJECT_ROOT is None:
    print("❌  Project source not found. Expected dataset at /kaggle/input/patient-doctor-bridge/")
    print("   Please add the 'patient-doctor-bridge' dataset to this notebook.")
    raise SystemExit("Dataset missing")
else:
    print(f"✅ Project root found at: {PROJECT_ROOT}")

# Add to Python path so imports resolve
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Make sure temp dirs exist
CACHE_DIR = pathlib.Path.home() / ".cache" / "pdb" / "models"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"   Whisper model cache: {CACHE_DIR}")

# Prescription sample images (in the project root)
SAMPLE_IMAGES = []
for i in range(1, 4):
    p = os.path.join(PROJECT_ROOT, f"Prescription{i}.png")
    if os.path.exists(p):
        SAMPLE_IMAGES.append(p)

print(f"   Sample prescription images found: {len(SAMPLE_IMAGES)}")
for img in SAMPLE_IMAGES:
    print(f"     • {os.path.basename(img)}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 5 — Start Ollama Daemon
# ═══════════════════════════════════════════════════════════════
import os
import subprocess
import time

import requests

OLLAMA_HOST = "http://localhost:11434"

def ollama_running():
    try:
        r = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=3)
        return r.status_code == 200
    except Exception:
        return False

if ollama_running():
    print("✅ Ollama is already running.")
else:
    print("Starting Ollama server…")
    env = os.environ.copy()
    env["OLLAMA_HOST"] = "0.0.0.0:11434"
    proc = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        env=env,
    )
    # Wait up to 30s for Ollama to be ready
    for i in range(30):
        time.sleep(1)
        if ollama_running():
            print(f"✅ Ollama started (took {i+1}s)")
            break
    else:
        print("⚠️  Ollama did not start in time — check logs")

# Verify
if ollama_running():
    print("   Ollama API reachable at", OLLAMA_HOST)
else:
    print("❌ Ollama is not reachable. Cannot continue.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 6 — Pull Gemma 4 Models
# Competition rule: ONLY gemma4:e2b and gemma4:e4b are permitted.
# ═══════════════════════════════════════════════════════════════
import subprocess
import time

REQUIRED_MODELS = ["gemma4:e2b", "gemma4:e4b"]

def model_available(tag: str) -> bool:
    r = subprocess.run(["ollama", "list"], capture_output=True, text=True)
    return tag in r.stdout

def pull_model(tag: str):
    if model_available(tag):
        print(f"✅ {tag} already available (skipping pull)")
        return
    print(f"⬇️  Pulling {tag}…  (this may take several minutes on first run)")
    r = subprocess.run(["ollama", "pull", tag], capture_output=True, text=True)
    if r.returncode == 0:
        print(f"✅ {tag} ready")
    else:
        print(f"❌ Failed to pull {tag}")
        print(r.stderr[-500:])

for model in REQUIRED_MODELS:
    pull_model(model)

print()
print("Installed models:")
subprocess.run(["ollama", "list"])


In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 7 — Hardware-Adaptive Configuration Report
# ═══════════════════════════════════════════════════════════════
from config.hardware import print_hardware_report

print_hardware_report()


In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 8 — Initialize Gemma 4 Engine + Warmup
# ═══════════════════════════════════════════════════════════════
from core.engine import GemmaEngine

print("Initializing GemmaEngine…")
engine = GemmaEngine()

print("Warming up models (loads both gemma4:e2b and gemma4:e4b into VRAM)…")
warmup_results = engine.warmup()
for model, ok in warmup_results.items():
    status = "✅ ready" if ok else "❌ unavailable"
    print(f"  {model}: {status}")

print()
print("✅ Engine ready. All inference runs locally via Ollama + Gemma 4.")


---
## 🔁 Scenario 1: Bridge Translation (Patient ↔ Doctor)

The core feature: a patient speaks in their native language; Gemma 4 translates to clinical English for the doctor. The doctor responds in English; Gemma 4 translates back to the patient's language.

**Supported languages:** Hindi · Telugu · Kannada · Tamil · English

> In production the text comes from Whisper ASR. Here we supply text directly to isolate the Gemma 4 translation quality.

In [ ]:
# ── Scenario 1a: Hindi Patient → Doctor English ──────────────────────────────
from rich.console import Console

from translation.service import TranslationService

console = Console()
svc = TranslationService(engine)

patient_text_hi = (
    "मुझे कल रात से बहुत तेज़ बुखार है। "
    "सिर में बहुत दर्द हो रहा है और पूरे शरीर में दर्द है। "
    "खाना भी नहीं खाया जा रहा।"
)

print("━" * 60)
print("PATIENT (Hindi):")
console.print(f"  [cyan]{patient_text_hi}[/cyan]")
print()

doctor_english = svc.patient_to_doctor(patient_text_hi, "hi")

print("TRANSLATION FOR DOCTOR (English):")
console.print(f"  [green]{doctor_english}[/green]")
print("━" * 60)


In [ ]:
# ── Scenario 1b: Doctor English → Hindi Patient ──────────────────────────────
doctor_response = (
    "You have viral fever. Take paracetamol 500 mg every 6 hours. "
    "Drink at least 3 litres of water a day. Rest completely for two days. "
    "Come back if the fever goes above 103°F or does not settle in three days."
)

print("━" * 60)
print("DOCTOR (English):")
console.print(f"  [cyan]{doctor_response}[/cyan]")
print()

patient_hindi = svc.doctor_to_patient(doctor_response, "hi")

print("TRANSLATION FOR PATIENT (Hindi):")
console.print(f"  [green]{patient_hindi}[/green]")
print("━" * 60)


In [ ]:
# ── Scenario 1c: Telugu Patient → Doctor English ─────────────────────────────
patient_text_te = (
    "నాకు రెండు రోజుల నుండి తీవ్రమైన కడుపు నొప్పి ఉంది. "
    "వాంతులు వస్తున్నాయి, జ్వరం కూడా ఉంది. "
    "కుడి వైపు కడుపులో ఎక్కువ నొప్పి ఉంది."
)

print("━" * 60)
print("PATIENT (Telugu):")
console.print(f"  [cyan]{patient_text_te}[/cyan]")
print()

doctor_english_te = svc.patient_to_doctor(patient_text_te, "te")

print("TRANSLATION FOR DOCTOR (English):")
console.print(f"  [green]{doctor_english_te}[/green]")
print("━" * 60)


In [ ]:
# ── Scenario 1d: Doctor English → Telugu Patient ─────────────────────────────
doctor_response_te = (
    "Based on your symptoms, we suspect appendicitis. "
    "We need to do a blood test and ultrasound immediately. "
    "Do not eat or drink anything until further notice. "
    "We will need to monitor you closely."
)

print("━" * 60)
print("DOCTOR (English):")
console.print(f"  [cyan]{doctor_response_te}[/cyan]")
print()

patient_telugu = svc.doctor_to_patient(doctor_response_te, "te")

print("TRANSLATION FOR PATIENT (Telugu):")
console.print(f"  [green]{patient_telugu}[/green]")
print("━" * 60)


In [ ]:
# ── Scenario 1e: Tamil Patient → Doctor English ──────────────────────────────
patient_text_ta = (
    "என்னுக்கு மூன்று நாட்களாக கடுமையான இருமல் இருக்கிறது. "
    "மூச்சு விடுவதில் சிரமம் இருக்கிறது. "
    "இரவில் வியர்வை அதிகமாக வருகிறது, சிறிது ரத்தமும் வந்தது."
)

print("━" * 60)
print("PATIENT (Tamil):")
console.print(f"  [cyan]{patient_text_ta}[/cyan]")
print()

doctor_english_ta = svc.patient_to_doctor(patient_text_ta, "ta")

print("TRANSLATION FOR DOCTOR (English):")
console.print(f"  [green]{doctor_english_ta}[/green]")
print()

# Doctor response back to Tamil
doctor_resp_ta = (
    "Your symptoms suggest possible tuberculosis. "
    "We need to do a chest X-ray, sputum test, and blood work urgently. "
    "Please cover your mouth when coughing. Wear this mask."
)
patient_tamil = svc.doctor_to_patient(doctor_resp_ta, "ta")

print("DOCTOR RESPONSE → Tamil:")
console.print(f"  [green]{patient_tamil}[/green]")
print("━" * 60)


---
## 🚨 Scenario 2: Emergency Triage Extraction

When a patient arrives in distress, Gemma 4 (`gemma4:e4b` with extended reasoning) extracts structured triage information from their speech — chief complaint, severity, symptoms, vitals, and whether immediate attention is needed.

In [ ]:
# ── Scenario 2a: Critical Triage — Hindi (Chest Pain / Cardiac) ──────────────
from rich.table import Table

from translation.triage import TriageService

triage_svc = TriageService(engine)

critical_text = (
    "मेरे सीने में बहुत तेज़ दर्द हो रहा है, बायीं बाँह में भी दर्द है। "
    "सांस नहीं आ रही। "
    "पसीना बहुत आ रहा है। "
    "यह दर्द आधे घंटे से है।"
)

print("━" * 60)
print("PATIENT (Hindi) — Emergency:")
console.print(f"  [red]{critical_text}[/red]")
print()
print("Extracting triage data with Gemma 4 (think=True)…")

result = triage_svc.extract(critical_text, "hi")

# Display as rich table
t = Table(title="Triage Result", show_header=True, header_style="bold red")
t.add_column("Field", style="dim", min_width=22)
t.add_column("Value")

severity_color = {"critical": "red", "severe": "red", "moderate": "yellow", "mild": "green"}.get(
    result["severity"], "white"
)

t.add_row("Chief Complaint",        result["chief_complaint"])
t.add_row("Severity",               f'[{severity_color}]{result["severity"].upper()}[/{severity_color}]')
t.add_row("Duration",               result["duration"])
t.add_row("Symptoms",               "\n".join(f"• {s}" for s in result["symptoms"]) or "—")
t.add_row("Vitals Mentioned",       "\n".join(f"• {v}" for v in result["vitals_mentioned"]) or "none")
t.add_row("Needs Immediate Attn",   "[bold red]YES ⚠[/bold red]" if result["needs_immediate_attention"] else "[green]No[/green]")
console.print(t)
print("━" * 60)


In [ ]:
# ── Scenario 2b: Severe Triage — Telugu (Head Injury / Neurological) ─────────
severe_text = (
    "నాకు అకస్మాత్తుగా తీవ్రమైన తలనొప్పి వచ్చింది, "
    "చూపు మబ్బుగా ఉంది. "
    "వాంతులు వస్తున్నాయి. "
    "కాళ్ళు బలహీనంగా ఉన్నాయి. "
    "ఒక గంట నుండి ఉంది."
)

print("━" * 60)
print("PATIENT (Telugu) — Severe:")
console.print(f"  [yellow]{severe_text}[/yellow]")
print()

result_te = triage_svc.extract(severe_text, "te")

t2 = Table(title="Triage Result", show_header=True, header_style="bold yellow")
t2.add_column("Field", style="dim", min_width=22)
t2.add_column("Value")

sev_col = {"critical": "red", "severe": "red", "moderate": "yellow", "mild": "green"}.get(
    result_te["severity"], "white"
)
t2.add_row("Chief Complaint",       result_te["chief_complaint"])
t2.add_row("Severity",              f'[{sev_col}]{result_te["severity"].upper()}[/{sev_col}]')
t2.add_row("Duration",              result_te["duration"])
t2.add_row("Symptoms",              "\n".join(f"• {s}" for s in result_te["symptoms"]) or "—")
t2.add_row("Vitals Mentioned",      "\n".join(f"• {v}" for v in result_te["vitals_mentioned"]) or "none")
t2.add_row("Needs Immediate Attn",  "[bold red]YES ⚠[/bold red]" if result_te["needs_immediate_attention"] else "[green]No[/green]")
console.print(t2)
print("━" * 60)


In [ ]:
# ── Scenario 2c: Moderate Triage — Kannada (Diabetic patient) ────────────────
moderate_text = (
    "ನನಗೆ ಸಕ್ಕರೆ ಕಾಯಿಲೆ ಇದೆ. "
    "ಇಂದು ಬೆಳಿಗ್ಗೆಯಿಂದ ತಲೆ ಸುತ್ತುತ್ತಿದೆ. "
    "ಕೈ ಕಾಲು ನಡುಗುತ್ತಿದೆ. "
    "ಬೆವರು ಹರಿಯುತ್ತಿದೆ. "
    "ಬೆಳಿಗ್ಗೆ ಊಟ ಮಾಡಿಲ್ಲ."
)

print("━" * 60)
print("PATIENT (Kannada) — Moderate (Hypoglycemia):")
console.print(f"  [yellow]{moderate_text}[/yellow]")
print()

result_kn = triage_svc.extract(moderate_text, "kn")

t3 = Table(title="Triage Result", show_header=True, header_style="bold cyan")
t3.add_column("Field", style="dim", min_width=22)
t3.add_column("Value")

sev3 = {"critical": "red", "severe": "red", "moderate": "yellow", "mild": "green"}.get(
    result_kn["severity"], "white"
)
t3.add_row("Chief Complaint",       result_kn["chief_complaint"])
t3.add_row("Severity",              f'[{sev3}]{result_kn["severity"].upper()}[/{sev3}]')
t3.add_row("Duration",              result_kn["duration"])
t3.add_row("Symptoms",              "\n".join(f"• {s}" for s in result_kn["symptoms"]) or "—")
t3.add_row("Vitals Mentioned",      "\n".join(f"• {v}" for v in result_kn["vitals_mentioned"]) or "none")
t3.add_row("Needs Immediate Attn",  "[bold red]YES ⚠[/bold red]" if result_kn["needs_immediate_attention"] else "[green]No[/green]")
console.print(t3)
print("━" * 60)


---
## 💊 Scenario 3: Emergency Reassurance Phrases

In critical moments, doctors need to instantly convey calming messages to patients who don't speak English. Gemma 4 (`gemma4:e2b`) translates pre-built reassurance phrases at maximum speed.

In [ ]:
# ── Scenario 3: Emergency Reassurance — All Languages ────────────────────────
from rich.table import Table

from translation.reassurance import REASSURANCE_PHRASES, ReassuranceService

reassure_svc = ReassuranceService(engine)

TARGET_LANGUAGES = ["hi", "te", "kn", "ta"]
LANG_NAMES = {"hi": "Hindi", "te": "Telugu", "kn": "Kannada", "ta": "Tamil"}

# Test all URGENT phrases across all 4 languages
urgent_phrases = [p for cat, p in REASSURANCE_PHRASES if cat == "URGENT"]

for lang_code in TARGET_LANGUAGES:
    t = Table(
        title=f"Reassurance → {LANG_NAMES[lang_code]}",
        show_header=True,
        header_style="bold green",
    )
    t.add_column("English (Original)", style="dim", min_width=40)
    t.add_column(f"{LANG_NAMES[lang_code]} Translation")

    for phrase in urgent_phrases:
        translated = reassure_svc.translate(phrase, lang_code)
        t.add_row(phrase, translated)

    console.print(t)
    print()


In [ ]:
# ── Scenario 3b: Full phrase bank for Hindi ───────────────────────────────────
print("Full reassurance phrase bank → Hindi:")
print("━" * 60)

full_table = Table(show_header=True, header_style="bold cyan")
full_table.add_column("Category", min_width=8)
full_table.add_column("English")
full_table.add_column("Hindi Translation")

cat_colors = {
    "URGENT": "red", "MEDICAL": "yellow", "COMFORT": "green", "INFO": "blue"
}

for cat, phrase in REASSURANCE_PHRASES:
    col = cat_colors.get(cat, "white")
    translated = reassure_svc.translate(phrase, "hi")
    full_table.add_row(
        f"[{col}]{cat}[/{col}]",
        phrase,
        translated,
    )

console.print(full_table)


---
## 📋 Scenario 4: Prescription OCR

Patients often can't read their own prescriptions (handwritten or in English). Gemma 4 (`gemma4:e4b`, multimodal vision) reads the prescription image and extracts a structured list of medicines with dosage, frequency, and instructions.

In [ ]:
# ── Scenario 4: Prescription OCR ─────────────────────────────────────────────
import os

from rich.table import Table

from translation.prescription import PrescriptionService

rx_svc = PrescriptionService(engine)

if not SAMPLE_IMAGES:
    print("⚠️  No sample prescription images found in project root.")
    print("   Expected: Prescription1.png, Prescription2.png, Prescription3.png")
else:
    for img_path in SAMPLE_IMAGES:
        img_name = os.path.basename(img_path)
        print(f"\n{'━'*60}")
        print(f"Processing: {img_name}")
        print(f"{'━'*60}")

        # Show the image
        from IPython.display import Image as IPyImage
        from IPython.display import display
        display(IPyImage(img_path, width=500))

        try:
            result = rx_svc.extract(img_path)

            # Meta info
            meta_table = Table(title=f"Prescription Metadata — {img_name}",
                               show_header=False)
            meta_table.add_column("Field", style="dim")
            meta_table.add_column("Value")
            meta_table.add_row("Doctor",  result["doctor_name"])
            meta_table.add_row("Patient", result["patient_name"])
            meta_table.add_row("Date",    result["date"])
            meta_table.add_row("Notes",   result["notes"] or "—")
            console.print(meta_table)

            # Medicines table
            if result["medicines"]:
                med_table = Table(title="Medicines Extracted", show_header=True,
                                  header_style="bold purple")
                med_table.add_column("#", min_width=2)
                med_table.add_column("Medicine", min_width=16)
                med_table.add_column("Dosage")
                med_table.add_column("Form")
                med_table.add_column("Frequency")
                med_table.add_column("Duration")
                med_table.add_column("Instructions")

                for i, m in enumerate(result["medicines"], 1):
                    med_table.add_row(
                        str(i), m["name"], m["dosage"], m["form"],
                        m["frequency"], m["duration"], m["instructions"],
                    )
                console.print(med_table)
            else:
                print("⚠️  No medicines extracted from this image.")

        except Exception as exc:
            print(f"❌ OCR failed for {img_name}: {exc}")


---
## 🌐 Full Interactive Web UI

The following cell starts the PatientDoctorBridge web server and exposes it via **ngrok** so you can open the full UI in your browser. All 4 scenarios (Bridge, Triage, Prescription, Reassure) are accessible with a real microphone or by uploading files.

> **Note:** You need a free ngrok auth token. Get one at https://dashboard.ngrok.com/signup  
> Set it with: `ngrok config add-authtoken YOUR_TOKEN`  
> Without a token, ngrok still works for short-lived sessions.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 9 — Start Flask Web Server (background thread)
# ═══════════════════════════════════════════════════════════════
import threading
import time

# Re-use the already-warmed engine (avoid double warmup)
import web.server as _ws
from config.settings import WEB_PORT
from web.server import create_app

_ws._engine = engine  # inject our pre-warmed engine

flask_app = create_app()

def _run_flask():
    flask_app.run(
        host="0.0.0.0",
        port=WEB_PORT,
        debug=False,
        use_reloader=False,
        threaded=True,
    )

flask_thread = threading.Thread(target=_run_flask, daemon=True)
flask_thread.start()
time.sleep(2)
print(f"✅ Flask server running on port {WEB_PORT}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 10 — Expose via ngrok (public browser URL)
# ═══════════════════════════════════════════════════════════════
from pyngrok import ngrok

from config.settings import WEB_PORT

# Optional: set your ngrok auth token for longer sessions
# ngrok.set_auth_token("YOUR_NGROK_TOKEN_HERE")

try:
    public_url = ngrok.connect(WEB_PORT, "http")
    # pyngrok returns an object; get the string URL
    url_str = public_url.public_url if hasattr(public_url, "public_url") else str(public_url)
    print("=" * 60)
    print("✅ PatientDoctorBridge is LIVE!")
    print()
    print(f"  🌐 Open in browser: {url_str}")
    print()
    print("  Tabs to test:")
    print("  • Bridge     — Record Patient / Doctor audio")
    print("  • Triage     — Emergency symptom extraction")
    print("  • Prescription — Upload prescription photo")
    print("  • Reassure   — Emergency phrase translation")
    print("=" * 60)
except Exception as e:
    print(f"⚠️  ngrok tunnel failed: {e}")
    print("   The server is still running at http://localhost:5000")
    print("   Use Kaggle's port forwarding or a local browser to access it.")
    url_str = f"http://localhost:{WEB_PORT}"


In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 11 — Display Embedded UI (inline preview)
# ═══════════════════════════════════════════════════════════════
from IPython.display import HTML, display

# Show the full app in an inline frame
# Note: microphone access may be blocked in iframe — open the link above
# in a new tab for full audio functionality.
display(HTML(f"""
<div style="border:2px solid #38bdf8;border-radius:12px;overflow:hidden;margin:10px 0;">
  <div style="background:#1e293b;padding:10px 16px;color:#38bdf8;font-family:sans-serif;font-weight:700;">
    PatientDoctorBridge — Live Demo &nbsp;
    <a href="{url_str}" target="_blank"
       style="background:#0ea5e9;color:white;padding:4px 12px;border-radius:6px;
              text-decoration:none;font-size:0.85rem;">
      Open in new tab ↗
    </a>
  </div>
  <iframe src="{url_str}" width="100%" height="700px"
          style="border:none;display:block;">
  </iframe>
</div>
"""))


---
## ✅ All Scenarios Demonstrated

| # | Scenario | Status |
|---|---|---|
| 1 | Bridge Translation (Hindi ↔ English) | ✅ Tested |
| 2 | Bridge Translation (Telugu ↔ English) | ✅ Tested |
| 3 | Bridge Translation (Tamil ↔ English) | ✅ Tested |
| 4 | Emergency Triage — Critical (Hindi chest pain) | ✅ Tested |
| 5 | Emergency Triage — Severe (Telugu neurological) | ✅ Tested |
| 6 | Emergency Triage — Moderate (Kannada diabetic) | ✅ Tested |
| 7 | Reassurance Phrases → Hindi, Telugu, Kannada, Tamil | ✅ Tested |
| 8 | Prescription OCR (sample images) | ✅ Tested |
| 9 | Full Web UI via ngrok | ✅ Live |

### Key technical highlights
- **Hardware-adaptive**: auto-selects GPU float16 or CPU int8 for Whisper
- **Privacy-first**: all inference local via Ollama; audio deleted immediately after transcription
- **Gemma 4 only**: `gemma4:e2b` for speed, `gemma4:e4b` for reasoning/vision
- **Resilient JSON parsing**: 4-strategy fallback handles truncated Gemma 4 output
- **Production-grade web UI**: WebM recording, language auto-detection, TTS playback
